In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from lib.data.dataloading import load_nursing_5_class
import torch
from torch import nn
from lib.config import *
import matplotlib.pyplot as plt
from lib.models import ResNetClassifierFiveClass
from lib.modules import optimization_loop_multi_class

In [186]:
WINSIZE = 1001
DEVICE = 'cuda:0'
nursing_trainloader, nursing_testloader = load_nursing_5_class(range(11,71), WINSIZE, test_size=0.2, batch_size=256, window=False)

Already downloaded


  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [00:03<00:00, 16.61it/s]


In [197]:
model = ResNetClassifierFiveClass(WINSIZE, 3, (16,32,64)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()
print(sum([p.numel() for p in model.parameters() if p.requires_grad]))

114455


In [204]:
# search for optimal resnet architecture
for i in [2,4,8,16,32,64]:
    for j in [2,4,8,16,32,64,128,256]:
        for k in [2,4,8,16,32,64,128,256,512,1028]:
            if j < i or k < j or k < i:
                print('skip')
                continue
            model = ResNetClassifierFiveClass(WINSIZE, 3, (i,j,k)).to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
            criterion = nn.CrossEntropyLoss()
            print(sum([p.numel() for p in model.parameters() if p.requires_grad]))
            
            optimization_loop_multi_class(
                model,
                nursing_trainloader,
                nursing_testloader,
                criterion,
                optimizer,
                epochs=100,
                device=DEVICE,
                patience=30,
                # outdir='dev/test',
                writer=f'runs/test-search/_w{WINSIZE}_{model.dims_str}'
            )

15813


: Epoch 99: Train Loss: 1.3983: Dev Loss: 1.3731: 100%|██████████| 100/100 [01:17<00:00,  1.29it/s]


16061


: Epoch 99: Train Loss: 1.1807: Dev Loss: 1.3231: 100%|██████████| 100/100 [01:13<00:00,  1.37it/s]


16965


...

Early stopping at epoch 74
297045


: Epoch 56: Train Loss: 0.6283: Dev Loss: 1.2544:  56%|█████▌    | 56/100 [01:10<00:55,  1.26s/it] 


Early stopping at epoch 56
1135445


: Epoch 28: Train Loss: 0.87603: Dev Loss: 1.1387:  29%|██▉       | 29/100 [01:03<02:34,  2.18s/it]


KeyboardInterrupt: 

In [198]:
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=300,
    device=DEVICE,
    patience=100,
    # outdir='dev/test',
    writer=f'runs/test-newresnet/test_w{WINSIZE}_{model.dims_str}'
)

: Epoch 114: Train Loss: 0.01515: Dev Loss: 2.4274:  38%|███▊      | 114/300 [01:57<03:11,  1.03s/it] 

Early stopping at epoch 114
